ПАЙПЛАЙН: слово → значение → семантический класс

что делаем:
- грузим датасет с определениями (слово → значение)
- переводим определения в векторы
- сопоставляем

1. потом подаем в код слово в контексте
например: "солдат держал в руках автомат"

2. энкодим

3. ищем ближайшую дефиницию (по косинусной близости)
→ определение "ручное автоматическое скорострельное оружие"

4. подбираем тег(и)
→ семантические теги "instr" и "tool" и "weapon"



---



# Грузим датасет с дефинициями



In [ ]:
!pip install torch transformers scikit-learn sentence-transformers

In [ ]:
from datasets import load_dataset

ds = load_dataset("snagbreac/russian-reverse-dictionary-train-data") # датасет со словарными определениями

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/15.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/294007 [00:00<?, ? examples/s]

In [ ]:
print(len(ds['train']))

294007


In [ ]:
print(ds['train'][1000])

{'word': 'адъютантский', 'definition': 'принадлежащий адъютанту', 'df': 'efremova'}


In [ ]:
from collections import defaultdict

# индексируем определения чтобы удобнее обращаться
#TODO выгрузить чтобы не надо было каждый раз индексировать
word2defs = defaultdict(list)

for entry in ds['train']:
    word2defs[entry['word']].append(entry['definition'])

In [ ]:
automat_definitions = word2defs.get('автомат')
print(automat_definitions)

['устройство, самостоятельно, без непосредственного участия человека выполняющее какие-л. действия или операции в соответствии с заранее заданной программой', 'перен. тот, кто поступает не думая, делает что-л. механически, машинально', 'перен. разг. оценка, полученная учащимся, студентом без дополнительной проверки приобретенных ими знаний и основанная на личной уверенности преподавателя в достаточности этих знаний', 'ручное автоматическое скорострельное оружие', 'разг. телефон-автомат', 'разновидность армейского оружия', 'индивидуальное огнестрельное оружие', 'устройство, производящее работу по заданной программе без непосредственного участия человека', 'аппарат, производящий некоторые действия без непосредственного участия человека', 'ручное скорострельное оружие', 'машина, после включения самостоятельно выполняющая ряд заданных операций']


In [ ]:
#TODO: почистить от помет (напр: разг. дает тег "speech" к дефиниции, НО при этом понижает точность наиболее вероятных тегов)

# Грузим эмбеддинги для перевода контекстов, дефиниций и тегов в вектор

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Теги → эмбеддинги

In [ ]:
TAGS = {
    "abstr": "абстрактные понятия, не имеющие физической формы",
    "age": "возраст и возрастные характеристики",
    "animal": "животные и живые существа животного мира",
    "appear": "появление, возникновение чего-либо",
    "artwork": "произведения искусства как объекты творчества",
    "art": "искусство как деятельность и культурная сфера",
    "be": "существование, бытие, факт наличия",
    "behav": "поведение и модели действий",
    "berry": "ягоды как биологический класс плодов",
    "changest": "изменение состояния или характеристик",
    "cin": "кинематограф, кино как индустрия и искусство",
    "class": "класс, категория или группа объектов",
    "cloth": "одежда и текстильные изделия",
    "color": "цветовые характеристики и восприятие цвета",
    "concr": "конкретные предметные объекты",
    "constr": "здания и строительные сооружения",
    "contact": "контактирование и взаимодействие через соприкосновение или связь",
    "degr": "степень, мера интенсивности или выраженности",
    "device": "высокотехнологичные устройства и автоматические системы",
    "dir": "направление движения или ориентации",
    "disappear": "исчезновение или прекращение существования",
    "disease": "болезни и патологические состояния",
    "dish": "посуда и предметы кухонного использования",
    "dist": "расстояние между объектами",
    "doc": "документы и текстовые официальные материалы",
    "dram": "театр и драматическое искусство",
    "emot": "эмоции и эмоциональные состояния",
    "etn": "этнонимы и названия народов",
    "ev": "оценка, результат оценивания",
    "famn": "фамилии как имена семейного происхождения",
    "food": "еда и пищевые продукты",
    "form": "форма и структурные характеристики",
    "fruit": "плоды растений",
    "furn": "мебель и предметы обстановки",
    "grain": "зерновые культуры и зерно",
    "group": "группы людей или объединения",
    "hum": "человек как биологический и социальный субъект",
    "humq": "качества человека и личностные характеристики",
    "impact": "воздействие одного объекта на другой",
    "instr": "инструменты как средства выполнения действий",
    "inter": "взаимодействие между объектами или субъектами",
    "kin": "родственные отношения",
    "light": "свет и световые явления",
    "max": "большая степень, максимум значения",
    "ment": "ментальная и когнитивная сфера",
    "min": "малое значение, минимум",
    "move": "передвижение и движение",
    "mus": "музыка как искусство и звуковая система",
    "neg": "негативная оценка или отрицательная характеристика",
    "org": "организации и институциональные структуры",
    "part": "часть целого",
    "patrn": "отчество как часть имени",
    "perc": "восприятие и перцептивные процессы",
    "physiol": "физиологические процессы живых организмов",
    "physq": "физические свойства объектов",
    "place": "место и пространственные объекты",
    "plant": "растения и ботанические объекты",
    "posit": "позитивная оценка или положительная характеристика",
    "poss": "посессивная сфера, принадлежность",
    "prof": "профессии и виды профессиональной деятельности",
    "propn": "имена собственные",
    "psych": "психика и психические процессы",
    "put": "помещения и локализованные пространства",
    "qtm": "кванты и дискретные единицы",
    "quant": "количество и количественные характеристики",
    "set": "множества и совокупности объектов",
    "size": "размер и пространственные характеристики",
    "sound": "звук и акустические явления",
    "speech": "речь и речевая деятельность",
    "speed": "скорость движения или изменения",
    "stuff": "вещества и материалы",
    "taste": "вкус как сенсорная характеристика",
    "temper": "температура и тепловые характеристики",
    "text": "текстовые объекты и письменные данные",
    "time": "время и временные характеристики",
    "tool": "инструменты и орудия труда",
    "topon": "топонимы и географические названия",
    "transp": "транспорт и средства передвижения",
    "unit": "единицы измерения",
    "weapon": "оружие и военные средства",
    "weather": "погодные явления",
    "weight": "вес и масса объектов"
}

In [ ]:
tag_names = list(TAGS.keys())
tag_texts = list(TAGS.values())

tag_embeddings = model.encode(tag_texts)

In [ ]:
# близости тегов и что weapon близко к tool и instr

from sklearn.metrics.pairwise import cosine_similarity

def tag_similarity(tag1, tag2):
    i = tag_names.index(tag1)
    j = tag_names.index(tag2)

    sim = cosine_similarity(
        [tag_embeddings[i]],
        [tag_embeddings[j]]
    )[0][0]

    return float(sim)

In [ ]:
print("weapon ~ tool:", tag_similarity("weapon", "tool"))
print("weapon ~ instr:", tag_similarity("weapon", "instr"))
print("tool ~ instr:", tag_similarity("tool", "instr"))

weapon ~ tool: 0.47061824798583984
weapon ~ instr: 0.4117741584777832
tool ~ instr: 0.7665427923202515




---



In [ ]:
# а если брать дефиниции из словаря а не короткие описания...

weapon = word2defs.get('оружие')
# в описании тега: оружие и военные средства
for i in weapon:
  print(i)

print('///')

instr_or_tool = word2defs.get('инструмент')
# в описании тега: инструменты и орудия труда // инструменты как средства выполнения действий
for j in instr_or_tool:
  print(j)

орудие нападения и защиты
перен. средство борьбы
то же, что: вооружение
общее название устройств и средств для нападения и защиты
средство борьбы
средство, технически пригодное для нападения или защиты
совокупность средств для ведения боя
совокупность средств для ведения войны
перен. основной элемент экипировки кого-либо
///
орудие - обычно ручное - для производства каких-л. работ
разг. музыкальный инструмент
перен. средство, применяемое для достижения или осуществления чего-л
орудие для работы
устар. политический акт


# Дефиниции → эмбеддинги + подбираем теги

In [ ]:
# получаем список тегов для дефиниции

def assign_tags(definition, threshold=0.4, top_k=5):

    def_emb = model.encode([definition])

    sims = cosine_similarity(def_emb, tag_embeddings)[0]

    results = []

    # отбираем теги по порогу
    for tag, score in zip(tag_names, sims):
        if score >= threshold:
            results.append((tag, float(score)))

    # сортировка по релевантности
    results.sort(key=lambda x: x[1], reverse=True)

    if top_k:
        results = results[:top_k]

    return results

In [ ]:
# TODO фильтровать по словарям? тут 3 вида
for d in automat_definitions:
    print("\nDEF:", d)
    print("TAGS:", assign_tags(d))


DEF: устройство, самостоятельно, без непосредственного участия человека выполняющее какие-л. действия или операции в соответствии с заранее заданной программой
TAGS: [('device', 0.5685820579528809), ('instr', 0.44109058380126953)]

DEF: перен. тот, кто поступает не думая, делает что-л. механически, машинально
TAGS: [('instr', 0.5550556182861328), ('ment', 0.502479612827301), ('concr', 0.4914413392543793), ('perc', 0.4842706322669983), ('device', 0.4780408442020416)]

DEF: перен. разг. оценка, полученная учащимся, студентом без дополнительной проверки приобретенных ими знаний и основанная на личной уверенности преподавателя в достаточности этих знаний
TAGS: [('ev', 0.5688983798027039), ('posit', 0.41605567932128906), ('humq', 0.40056541562080383)]

DEF: ручное автоматическое скорострельное оружие
TAGS: [('weapon', 0.6501611471176147), ('instr', 0.47591692209243774), ('device', 0.4706840217113495), ('tool', 0.4248921275138855)]

DEF: разг. телефон-автомат
TAGS: [('device', 0.48724335432

In [ ]:
def1 = "разг. телефон-автомат"
print(assign_tags(def1, top_k=10))

def2 = "телефон-автомат"
print(assign_tags(def2, top_k=10))

[('device', 0.4872433543205261)]
[('device', 0.4513041079044342)]


# Парсим конлю

In [ ]:
!pip install conllu

In [ ]:
import re
SEM_PATTERN = re.compile(r"sem=\[([^\]]+)\]")

def extract_sem_tags(misc):
    """
    Универсальный извлекатель sem-тегов из MISC
    поддерживает dict и строку
    """
    if misc is None:
        return []

    # случай: conllu вернул dict
    if isinstance(misc, dict):
        sem_val = misc.get("sem")
        if not sem_val:
            return []
        sem_val = sem_val.strip("[]")
        return [t.strip() for t in sem_val.split(",")]

    # случай: строка
    match = SEM_PATTERN.search(str(misc))
    if not match:
        return []

    return [t.strip() for t in match.group(1).split(",")]

In [ ]:
from conllu import parse

def parse_conllu(text):
    sentences = parse(text)
    parsed = []

    for sent in sentences:
        tokens = []
        for token in sent:
            sem_tags = extract_sem_tags(token.get("misc"))

            tokens.append({
                "form": token["form"],
                "lemma": token["lemma"],
                "upos": token["upos"],
                "feats": token["feats"],
                "head": token["head"],
                "deprel": token["deprel"],
                "sem": sem_tags
            })

        parsed.append(tokens)

    return parsed

In [ ]:
text = '''
# sent_id = GramEval2020-test-fiction-2
# genre = fiction
# text = Может быть, потому, что музыкальный автомат грохотал здесь не так оглушительно, как везде.
1	Может	мочь	VERB	_	Aspect=Imp|Mood=Ind|Number=Sing|Person=3|Tense=Pres|Transit=Intr|VerbForm=Fin|Voice=Act	4	parataxis:discourse	_	sem=[abstr]|hls
2	быть	быть	VERB	_	Transit=Intr|VerbForm=Inf|Voice=Act	1	xcomp	_	SpaceAfter=No
3	,	,	PUNCT	_	_	1	punct	_	_
4	потому	потому	ADVPRO	_	Degree=Pos|PronType=Dem	0	root	_	SpaceAfter=No
5	,	,	PUNCT	_	_	9	punct	_	_
6	что	что	SCONJ	_	_	9	mark	_	_
7	музыкальный	музыкальный	ADJ	_	Case=Nom|Degree=Pos|Gender=Masc|Number=Sing	8	amod	_	sem=[mus,sound]|hls
8	автомат	автомат	NOUN	_	Animacy=Inan|Case=Nom|Gender=Masc|Number=Sing	9	nsubj	_	sem=[concr,device,tool]|hls
9	грохотал	грохотать	VERB	_	Aspect=Imp|Gender=Masc|Mood=Ind|Number=Sing|Tense=Past|Transit=Intr|VerbForm=Fin|Voice=Act	4	advcl	_	sem=[sound]|hls
10	здесь	здесь	ADVPRO	_	Degree=Pos|PronType=Dem	9	advmod	_	sem=[place]|hls
11	не	не	PART	_	Polarity=Neg	12	advmod	_	_
12	так	так	ADVPRO	_	Degree=Pos|PronType=Dem	13	advmod	_	sem=[degr]|hls
13	оглушительно	оглушительно	ADV	_	Degree=Pos	9	advmod	_	SpaceAfter=No|sem=[sound]|hls
14	,	,	PUNCT	_	_	16	punct	_	_
15	как	как	SCONJ	_	_	16	mark	_	_
16	везде	везде	ADVPRO	_	Degree=Pos|PronType=Tot	12	advcl	_	SpaceAfter=No|sem=[place]|hls
17	.	.	PUNCT	_	_	4	punct	_	_
'''

parsed_text = parse_conllu(text)[0]
for token in parsed_text:
  print(token)
  print(token['sem'])

{'form': 'Может', 'lemma': 'мочь', 'upos': 'VERB', 'feats': {'Aspect': 'Imp', 'Mood': 'Ind', 'Number': 'Sing', 'Person': '3', 'Tense': 'Pres', 'Transit': 'Intr', 'VerbForm': 'Fin', 'Voice': 'Act'}, 'head': 4, 'deprel': 'parataxis:discourse', 'sem': ['abstr']}
['abstr']
{'form': 'быть', 'lemma': 'быть', 'upos': 'VERB', 'feats': {'Transit': 'Intr', 'VerbForm': 'Inf', 'Voice': 'Act'}, 'head': 1, 'deprel': 'xcomp', 'sem': []}
[]
{'form': ',', 'lemma': ',', 'upos': 'PUNCT', 'feats': None, 'head': 1, 'deprel': 'punct', 'sem': []}
[]
{'form': 'потому', 'lemma': 'потому', 'upos': 'ADVPRO', 'feats': {'Degree': 'Pos', 'PronType': 'Dem'}, 'head': 0, 'deprel': 'root', 'sem': []}
[]
{'form': ',', 'lemma': ',', 'upos': 'PUNCT', 'feats': None, 'head': 9, 'deprel': 'punct', 'sem': []}
[]
{'form': 'что', 'lemma': 'что', 'upos': 'SCONJ', 'feats': None, 'head': 9, 'deprel': 'mark', 'sem': []}
[]
{'form': 'музыкальный', 'lemma': 'музыкальный', 'upos': 'ADJ', 'feats': {'Case': 'Nom', 'Degree': 'Pos', 'Gend

### Собираем словоупотребоения по группам тегов

In [ ]:
#TODO: доделать

def extract_examples(conllu_text, target_tags=None):
    sentences = parse(conllu_text)
    results = []

    if target_tags:
        target_tags = [t.strip().lower() for t in target_tags]

    for sent in sentences:
        sentence_text = sent.metadata.get("text", "")

        for token in sent:
            sem_tags = extract_sem_tags(token.get("misc"))
            sem_tags = [t.lower() for t in sem_tags]

            if not sem_tags:
                continue

            if target_tags:
                if not any(tag in sem_tags for tag in target_tags):
                    continue

            results.append({
                "form": token["form"],
                "lemma": token["lemma"],
                "upos": token["upos"],
                "sem": sem_tags,
                "sentence": sentence_text,
                "token_id": token["id"]
            })

    return results

In [ ]:
target_tags = ["abstr", "mus", "sound", "concr", "device", "tool", "degr", "place"]

finding_tags = extract_examples(text, target_tags)
for found_tags in finding_tags:
  print(found_tags)

{'form': 'Может', 'lemma': 'мочь', 'upos': 'VERB', 'sem': ['abstr'], 'sentence': 'Может быть, потому, что музыкальный автомат грохотал здесь не так оглушительно, как везде.', 'token_id': 1}
{'form': 'музыкальный', 'lemma': 'музыкальный', 'upos': 'ADJ', 'sem': ['mus', 'sound'], 'sentence': 'Может быть, потому, что музыкальный автомат грохотал здесь не так оглушительно, как везде.', 'token_id': 7}
{'form': 'автомат', 'lemma': 'автомат', 'upos': 'NOUN', 'sem': ['concr', 'device', 'tool'], 'sentence': 'Может быть, потому, что музыкальный автомат грохотал здесь не так оглушительно, как везде.', 'token_id': 8}
{'form': 'грохотал', 'lemma': 'грохотать', 'upos': 'VERB', 'sem': ['sound'], 'sentence': 'Может быть, потому, что музыкальный автомат грохотал здесь не так оглушительно, как везде.', 'token_id': 9}
{'form': 'здесь', 'lemma': 'здесь', 'upos': 'ADVPRO', 'sem': ['place'], 'sentence': 'Может быть, потому, что музыкальный автомат грохотал здесь не так оглушительно, как везде.', 'token_id': 



---



к этому моменту у нас есть:

* готовый эмбеддинг (взяли paraphrase-multilingual-MiniLM-L12-v2) - заменить?
*   словарик с тегами
*   векторизованные теги
* датасет с дефинициями (взяли russian-reverse-dictionary-train-data)
* векторизованные дефиниции
* код для подбора наиболее близких тегов к дефинициям

что надо дальше: по контексту слова подбираем, какая из дефиниций больше всего подходит, и потом на основе этой дефиниции к слову в контексте присваивать теги

TODO: посмотреть семинар про ADAGRAM


#